# Expanded_Eurofer_CD — Cluster Dynamics for bcc Fe / EUROFER97

Full per-size cluster dynamics model implementing:
- **Master equations** (Eqs. 152, 155, 157) for SIA clusters, He-vacancy bubbles, and free He.
- **He-reduction**: Case 2 fission (Eq. 175, decoupled) or Case 1 fusion (Eq. 174, mean-field).
- **Size-bin moments** (Chapter 9, Eqs. 188–211) for large-N efficiency.
- **Solute trapping** in EUROFER97 (Cr, W, Mn, C, N) via effective diffusivities (Eqs. 42, 48, 52).
- **1D/3D mixed transport** for glissile SIA clusters (Eq. 141).

Physics reference: Ghoniem, N.M. (2026), *'A Cluster Dynamics Model for Radiation Damage Evolution in Ferritic-Martensitic Steels'* (Rate_Equations.pdf).

---

## Solver modes
| `solver_mode` | Description |
|---|---|
| `cpp_full` | C++ CVODE BDF, full system, dense/band/GMRES |
| `cpp_sliding_win` | C++ CVODE BDF with sliding SIA window (Phase III) |
| `sliding_OpenMP` | Sliding window + OpenMP intra-RHS parallelism (Phase IV) |

## Physics options
| `physics_option` | Equations | He reduction | Cascade |
|---|---|---|---|
| `full_CD_fission` | Eqs. 152, 155, 157 | Case 2, Eq. 175 (decoupled) | Fission (Table 2) |
| `full_CD_fusion` | Eqs. 152, 155, 157 | Case 1, Eq. 174 (mean-field) | Fusion (Table 2) |
| `bin_moment_CD_fission` | Chapter 9 moments | Case 2 | Fission |
| `bin_moment_CD_fusion` | Chapter 9 moments | Case 1 | Fusion |

In [1]:
# ── Environment setup ─────────────────────────────────────────────────────────
import sys, os
from pathlib import Path

# Add Expanded_Eurofer_CD root to path
MODULE_ROOT = Path('..').resolve()
REPO_ROOT   = MODULE_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
if str(MODULE_ROOT) not in sys.path:
    sys.path.insert(0, str(MODULE_ROOT))

print(f'Module root: {MODULE_ROOT}')
print(f'Repo root:   {REPO_ROOT}')

Module root: C:\Users\Owner\Documents\Repos\EuroferMicrostructure\Expanded_Eurofer_CD
Repo root:   C:\Users\Owner\Documents\Repos\EuroferMicrostructure


## 1. Generate Excel input file (run once)

In [ ]:
# Generate input/input_parameters.xlsx from Rate_Equations.pdf parameters
import subprocess
result = subprocess.run(
    [sys.executable, str(MODULE_ROOT / 'create_excel.py')],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('ERROR:', result.stderr)

## 2. Build C++ solver (run once after code changes)

In [ ]:
# Build the C++ SUNDIALS solver
build_dir = MODULE_ROOT / 'build'
build_dir.mkdir(exist_ok=True)

cmake_src  = MODULE_ROOT / 'cpp_utils'
cmake_cmd  = ['cmake', '-S', str(cmake_src), '-B', str(build_dir),
               '-DCMAKE_BUILD_TYPE=Release']
build_cmd  = ['cmake', '--build', str(build_dir), '--config', 'Release']

for cmd in [cmake_cmd, build_cmd]:
    res = subprocess.run(cmd, capture_output=True, text=True)
    print(' '.join(cmd))
    if res.returncode != 0:
        print('STDERR:', res.stderr[-2000:])
    else:
        print('OK')
        print(res.stdout[-500:])

In [ ]:
# ── Environment setup ─────────────────────────────────────────────────────────
import sys, os
from pathlib import Path

MODULE_ROOT = Path('..').resolve()
REPO_ROOT   = MODULE_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
if str(MODULE_ROOT) not in sys.path:
    sys.path.insert(0, str(MODULE_ROOT))

print(f'Module root: {MODULE_ROOT}')
print(f'Repo root:   {REPO_ROOT}')


## 2. Select solver mode and physics option
# ── User selections ───────────────────────────────────────────────────────────
SOLVER_MODE    = 'cpp_full'
PHYSICS_OPTION = 'full_CD_fission'

N_CLUSTERS = 50    # max SIA cluster size  (Eq. 152)
M_CLUSTERS = 50    # max vacancy cluster size (Eq. 155)

# Concentration floor: concentrations below C_FLOOR are clamped to C_FLOOR
# before evaluating rate terms; derivatives of floor-clamped variables are
# also clamped to >= 0 to prevent the solver driving them negative.
C_FLOOR = 1e-20     # [atom fraction]

# Free helium treatment:
#   'dynamic'            — integrate c_h as a full ODE variable (Eq. 157)
#   'quasi_steady_state' — eliminate c_h from state; compute algebraically from
#                          dc_h/dt = 0 at each RHS call (E_m_h = 0.06 eV → fast)
HE_OPTIONS = 'quasi_steady_state'

print(f'solver_mode:    {SOLVER_MODE}')
print(f'physics_option: {PHYSICS_OPTION}')
print(f'N={N_CLUSTERS}  M={M_CLUSTERS}')
print(f'C_floor:        {C_FLOOR:.1e}')
print(f'he_options:     {HE_OPTIONS}')


## 3. ── Solver configuration ──────────────────────────────────────────────────

SOLVER_CONFIG = {
    't_span':   (1e-5, 1e3),     # [s]  irradiation time range
    'n_points': 200,
    'log_time': True,
    'rtol':     1e-4,
    'atol':     1e-20,
    'solver_method': {
        'backend':           'cvode',
        'lmm':               'bdf',
        'linsol':            'gmres',
        'window_w0_i':       50,
        'window_width':      200,
        'window_C_expand':   1e-18,
        'window_expand_pad': 10,
        'window_omp_threads':0,
        'window_prec':       1,
        'window_gmres_maxl': 20,
        'window_N_thresh':   500,
    }
}


## 4. Inspect parameters
from Expanded_Eurofer_CD.py_utils.simulation import ExpandedEuroferCDSimulation

sim = ExpandedEuroferCDSimulation(
    N=N_CLUSTERS, M=M_CLUSTERS,
    solver_mode=SOLVER_MODE,
    physics_option=PHYSICS_OPTION,
    C_floor=C_FLOOR,
    he_options=HE_OPTIONS,
)
sim.input_data.display_parameters()


## 5. Run simulation
results = sim.run(solver_config=SOLVER_CONFIG, save_output=True)

if results is not None:
    t = results['t']
    print(f'Solution: {len(t)} time points, t=[{t[0]:.2e}, {t[-1]:.2e}] s')
    print(f'Final dose: {results["dose"][-1]:.3f} dpa')
    print(f'Swelling (final): {results["swelling"][-1]*100:.4f} %')
    print(f'C_He_tot (final): {results["C_He_tot"][-1]:.3e}')
    print(f'delta_FP (final): {results["delta_FP"][-1]:.3e}  (Eq. 164)')
    print(f'delta_He (final): {results["delta_He"][-1]:.3e}  (Eq. 165)')
else:
    print('Simulation failed — check C++ build and parameter file.')


## 6. Plots
%matplotlib inline
from Expanded_Eurofer_CD.py_utils import visualization as viz

if results is not None:
    label = f'{SOLVER_MODE}/{PHYSICS_OPTION}/{HE_OPTIONS}'
    viz.plot_point_defects(results, title=label)
    viz.plot_totals(results, title=label)
    viz.plot_swelling(results, title=label)
    viz.plot_mean_sizes(results, title=label)
    viz.plot_he_content(results, title=label)
    viz.plot_conservation(results, title=label)
    viz.plot_number_densities(results, title=label)
    viz.plot_size_distribution(results, sim.input_data, title=label)
    viz.plot_sia_clusters(results, sim.input_data, title=label)
    viz.plot_vac_clusters(results, sim.input_data, title=label)


Module root: C:\Users\Owner\Documents\Repos\EuroferMicrostructure\Expanded_Eurofer_CD
Repo root:   C:\Users\Owner\Documents\Repos\EuroferMicrostructure
solver_mode:    cpp_full
physics_option: full_CD_fission
N=50  M=50
C_floor:        1.0e-20
he_options:     quasi_steady_state
Initializing Expanded_Eurofer_CD simulation…
Loading parameters from: C:\Users\Owner\Documents\Repos\EuroferMicrostructure\Expanded_Eurofer_CD\input\input_parameters.xlsx
Successfully loaded all five parameter sheets.
Derived: T=600.0 K  Cv_eq=1.588e-17  Di_eff=5.345e-11  Dv_eff=1.045e-13 m2/s  spectrum='fission'
ReactionRates: K_SIA_grow[0]=8.039e+09  K_VAC_grow[0]=1.571e+07  G_VAC[0]=0.000e+00  K_iv=4.387e+07
  k2_SIA[0]=5.880e+01  k2_vac=1.045e-01  k2_He=7.727e+04
RateEquations: he_mode='case2'  he_options='quasi_steady_state'  N_eq=101  N=50  M=50  G_He=7.500e-13 at.frac/s  C_floor=1.0e-20  beta_He=4.508e-08
Simulation initialized: solver_mode='cpp_full'  physics_option='full_CD_fission'

ENERGETICS
  a: 2.8

Expanded_Eurofer_CD solver: N_eq=101  physics_option=0  he_mode=0  he_options=quasi_steady_state  C_floor=1e-20  window_mode=0
  [diag] t=1.097e-05  c_i1=1.229e-13  c_v1=2.473e-13  c_i2=1.238e-14  c_i5=2.358e-15  c_v2=7.751e-15  c_v5=7.844e-16  Q_tot=1.001e-20  SIA_tot=2.243e-13  VAC_tot=2.910e-13
  [diag] t=1.203e-05  c_i1=2.598e-13  c_v1=5.186e-13  c_i2=3.236e-14  c_i5=2.420e-15  c_v2=1.625e-14  c_v5=1.645e-15  Q_tot=1.006e-20  SIA_tot=4.382e-13  VAC_tot=6.101e-13
  [diag] t=1.320e-05  c_i1=4.133e-13  c_v1=8.162e-13  c_i2=5.812e-14  c_i5=2.420e-15  c_v2=2.558e-14  c_v5=2.589e-15  Q_tot=1.016e-20  SIA_tot=6.607e-13  VAC_tot=9.603e-13
  [diag] t=1.448e-05  c_i1=5.859e-13  c_v1=1.143e-12  c_i2=8.598e-14  c_i5=2.420e-15  c_v2=3.581e-14  c_v5=3.624e-15  Q_tot=1.031e-20  SIA_tot=8.966e-13  VAC_tot=1.344e-12
  [diag] t=1.589e-05  c_i1=7.800e-13  c_v1=1.501e-12  c_i2=1.135e-13  c_i5=2.420e-15  c_v2=4.704e-14  c_v5=4.760e-15  Q_tot=1.054e-20  SIA_tot=1.149e-12  VAC_tot=1.766e-12
  [diag] t=1.

: 

## 7. Compare fission vs. fusion spectra

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

results_all = {}
for po in ['full_CD_fission', 'full_CD_fusion']:
    print(f'\nRunning {po}...')
    sim_po = ExpandedEuroferCDSimulation(
        N=N_CLUSTERS, M=M_CLUSTERS,
        solver_mode='cpp_full',
        physics_option=po
    )
    res_po = sim_po.run(solver_config=SOLVER_CONFIG, save_output=False)
    if res_po is not None:
        results_all[po] = res_po
        print(f'  swelling={res_po["swelling"][-1]*100:.4f}%')

if len(results_all) == 2:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    colors = {'full_CD_fission': 'steelblue', 'full_CD_fusion': 'tomato'}
    labels = {'full_CD_fission': 'Fission (Case 2, Eq. 175)',
              'full_CD_fusion':  'Fusion (Case 1, Eq. 174)'}

    for po, res in results_all.items():
        dose = res['dose']
        c = colors[po]; lb = labels[po]
        axes[0].loglog(dose, res['C_SIA_tot'], color=c, label=lb)
        axes[1].loglog(dose, res['C_VAC_tot'], color=c, label=lb)
        axes[2].semilogx(dose, res['swelling']*100, color=c, label=lb)

    axes[0].set(xlabel='Dose (dpa)', ylabel='SIA content', title='SIA Total (Σn·cn)')
    axes[1].set(xlabel='Dose (dpa)', ylabel='Vac content', title='Vacancy Total (Σm·cm)')
    axes[2].set(xlabel='Dose (dpa)', ylabel='Swelling (%)', title='Void Swelling (Eq. 161)')
    for ax in axes: ax.legend(); ax.grid(True, alpha=0.3)
    fig.tight_layout()
    plt.show()

## 8. Compare full_CD vs. bin_moment (fission)

In [ ]:
results_cmp = {}
for po in ['full_CD_fission', 'bin_moment_CD_fission']:
    print(f'\nRunning {po}...')
    sim_po = ExpandedEuroferCDSimulation(
        N=N_CLUSTERS, M=M_CLUSTERS,
        solver_mode='cpp_full',
        physics_option=po
    )
    res_po = sim_po.run(solver_config=SOLVER_CONFIG, save_output=False)
    if res_po is not None:
        results_cmp[po] = res_po
        re = sim_po.rate_equations
        if hasattr(re, 'K'):
            print(f'  Bin-moment: K_bins={re.K}  N_eq={re.N_eq}')

if len(results_cmp) == 2:
    fig, ax = plt.subplots(figsize=(8, 5))
    for po, res in results_cmp.items():
        ax.semilogx(res['dose'], res['swelling']*100, label=po)
    ax.set(xlabel='Dose (dpa)', ylabel='Swelling (%)',
           title='Full CD vs. Bin-Moment (fission, Eq. 161)')
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.show()

## 9. Inspect cascade production spectrum (Eqs. 7-10, Table 2)

In [ ]:
from Expanded_Eurofer_CD.py_utils.defect_production import production_rates, FISSION, FUSION
import numpy as np, matplotlib.pyplot as plt

G_test = 1e-6   # dpa/s
N_test, M_test = 200, 200

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for spec, color, label in [
        ('fission', 'steelblue', 'Fission (Table 2)'),
        ('fusion',  'tomato',    'Fusion (Table 2)')]:
    Pr_i, Pr_v, G_He = production_rates(G_test, spec, N_test, M_test)
    ns = np.arange(1, N_test + 1)
    ms = np.arange(1, M_test + 1)
    axes[0].semilogy(ns, Pr_i[1:], color=color, label=label)
    axes[1].semilogy(ms, Pr_v[1:], color=color, label=label)

axes[0].set(xlabel='SIA cluster size n', ylabel=r'$P_n^i$ [s$^{-1}$]',
            title='SIA Cascade Production (Eq. 12)')
axes[1].set(xlabel='Vacancy cluster size m', ylabel=r'$P_m^v$ [s$^{-1}$]',
            title='Vacancy Cascade Production (Eq. 13)')
for ax in axes: ax.legend(); ax.grid(True, which='both', alpha=0.3)
fig.tight_layout()
plt.show()

print('Fission parameters (Table 2):', FISSION)
print('Fusion  parameters (Table 2):', FUSION)

## 10. Binding energy inspection (Eqs. 66-108, Tables 18-19)

In [ ]:
from Expanded_Eurofer_CD.py_utils.binding_energies import (
    E_b_void, E_b_loop_i, E_b_loop_v, E_b_bubble
)
import numpy as np, matplotlib.pyplot as plt

# Parameters from Table 5
E_f_v   = 2.0       # eV
gamma_s = 2.0       # J/m^2
Omega   = 1.18e-29  # m^3
T       = 600.0     # K

ms = np.arange(1, 101)
ns = np.arange(2, 101)

Eb_void = np.array([E_b_void(m, E_f_v, gamma_s, Omega) for m in ms])
Eb_loop = E_b_loop_i(ns)   # uses Table 18 defaults

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].semilogx(ms, Eb_void, color='tomato')
axes[0].set(xlabel='Vacancy cluster size m', ylabel='E_b^void [eV]',
            title='Void Binding Energy (Eqs. 66-67, Table 18)')
axes[1].semilogx(ns, Eb_loop, color='steelblue')
axes[1].set(xlabel='SIA loop size n', ylabel='E_b^loop [eV]',
            title='Interstitial Loop Binding (Eqs. 106-108, Table 18)')
for ax in axes: ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()